In [ ]:
import pandas as pd
import json

In [1]:
def simple_json_to_excel(json_file, csv_file='vacancies.csv'):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    vacancies = data.get('items', [])
    
    def get_field_value(data, field_path):
        if isinstance(data, dict):
            value = data.get(field_path, '')
        else:
            value = ''
        
        if isinstance(value, dict):
            return value.get('name', '')
        elif isinstance(value, list):
            names = []
            for item in value:
                if isinstance(item, dict) and 'name' in item:
                    names.append(item['name'])
            return ', '.join(names)
        else:
            return value
    
    parsed_vacancies = []
    seen_ids = set()
    
    for v in vacancies:
        vac_id = v.get('id', '')

        if vac_id in seen_ids:
            continue
        seen_ids.add(vac_id)
        
        parsed = {
            'ID': vac_id,
            'Название': v.get('name', ''),
            'Департамент': get_field_value(v, 'department'),
            'Город': get_field_value(v, 'area'),
            'Опыт': get_field_value(v, 'experience'),
            'Расписание': get_field_value(v, 'schedule'),
            'Формат работы': get_field_value(v, 'work_format'),
            'Рабочие часы': get_field_value(v, 'working_hours'),
            'График': get_field_value(v, 'work_schedule_by_days'),
            'Зарплата': 'Не указана',
            'Ссылка': v.get('alternate_url', ''),
            'Требования': v.get('snippet', {}).get('requirement', '') if isinstance(v.get('snippet'), dict) else '',
            'Зона ответственности': v.get('snippet', {}).get('responsibility', '') if isinstance(v.get('snippet'), dict) else ''
        }

        salary = v.get('salary')
        if salary and isinstance(salary, dict):
            salary_from = salary.get('from')
            salary_to = salary.get('to')
            currency = salary.get('currency', 'RUR')
            
            if salary_from is not None and salary_to is not None:
                parsed['Зарплата'] = f"{salary_from:,} - {salary_to:,}".replace(',', ' ')
            elif salary_from is not None:
                parsed['Зарплата'] = f"от {salary_from:,}".replace(',', ' ')
            elif salary_to is not None:
                parsed['Зарплата'] = f"до {salary_to:,}".replace(',', ' ')
        
        parsed_vacancies.append(parsed)
    
    df = pd.DataFrame(parsed_vacancies)
    df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    
    print(f"Сохранено {len(df)} вакансий в {csv_file}")
    return df

df = simple_json_to_excel('vacancies.json', csv_file='vacancies.csv')

Сохранено 5 вакансий в vacancies.csv
